In [1]:
# アクセス列の分割
import pandas as pd
import re

In [2]:
# CSVファイルを読み込み
df1 = pd.read_csv('test.csv')

In [3]:
df = df1[['id', 'アクセス']].copy()

In [4]:
# アクセス情報を分割
df.loc[:, 'アクセス_分割'] = df['アクセス'].str.split('\t\t')

In [5]:
# 各駅ごとの情報を展開
for i in range(3):  # 最大で3つの駅があるとして処理
    df.loc[:, f'路線{i+1}'] = df['アクセス_分割'].apply(lambda x: x[i] if len(x) > i else None)

In [6]:
# 駅情報を分解する関数
def split_station_info(station_info):
    if station_info:
        parts = station_info.split('\t')
        if len(parts) == 3:
            return parts[0], parts[1], parts[2].replace('徒歩', '').replace('分', '')  # 路線, 駅名, 徒歩分数
    return None, None, None

In [7]:
# 各路線・駅・徒歩分数を分解して新しい列に格納
for i in range(3):
    df[f'路線{i+1}_名'], df[f'駅{i+1}_名'], df[f'徒歩{i+1}_分'] = zip(*df[f'路線{i+1}'].map(split_station_info))

In [8]:
# 不要な中間列を削除
df.drop(columns=['アクセス', 'アクセス_分割', '路線1', '路線2', '路線3'], inplace=True)

In [9]:
# バスの情報を含む列の値を置き換える関数
def convert_bus_to_walking_time(val):
    # val が文字列でない場合、そのまま返す
    if not isinstance(val, str):
        return val

    # バスの移動時間（例: バス(23)）
    match = re.search(r'バス\((\d+)\)', val)
    if match:
        # バスの分数を5倍にして、他の部分を取り除く
        bus_minutes = int(match.group(1)) * 5
        return str(bus_minutes)

    # バスの移動時間（例: バス(23分)）
    match = re.search(r'バス\((\d+)分\)', val)
    if match:
        # バスの分数を5倍にして、他の部分を取り除く
        bus_minutes = int(match.group(1)) * 5
        return str(bus_minutes)

    return val

In [10]:
# '徒歩1_分'列の値を変換
df.loc[:, '徒歩1_分'] = df['徒歩1_分'].apply(convert_bus_to_walking_time)

In [11]:
# '徒歩2_分'列の値を変換
df.loc[:, '徒歩2_分'] = df['徒歩2_分'].apply(convert_bus_to_walking_time)

In [12]:
# '徒歩3_分'列の値を変換
df.loc[:, '徒歩3_分'] = df['徒歩3_分'].apply(convert_bus_to_walking_time)

In [13]:
df.to_csv('test_accese.csv', index=False)